In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pylupnt as pnt

np.set_printoptions(formatter={"float": "{: 0.3f}".format})

In [ ]:
# Frames:
# - Moon Centered Inertial (CI) Frame

# Classical Orbital Elements *********************************
sma = 5740e3  # [m] Semi-major axis
ecc = 0.58  # [-] Eccentricity
inc = 54.856 * pnt.RAD  # [deg] Inclination
raan = 25 * pnt.RAD  # [rad] Right ascension of the ascending node
aop = 86.322 * pnt.RAD  # [rad] Argument of periapsis
ma = 340 * pnt.RAD  # [rad] Mean anomaly

# Classical Orbital Elements (sma, ecc, inc, raan, aop, ma) [km, -, rad]
coe0_ci = np.array([sma, ecc, inc, raan, aop, ma])
# Position and Velocity (x, y, z, vx, vy, vz) [km, km/s]
rv0_ci = pnt.classical_to_cart(coe0_ci, pnt.GM_MOON)
period = pnt.get_orbital_period(sma, pnt.GM_MOON)

print("Period:", round(period / pnt.SECS_HOUR, 2), "hours")
print("Satellite:", rv0_ci)

# User Position ********************************************
lat = 20 * pnt.RAD  # [rad] Latitude
lon = 50 * pnt.RAD  # [rad] Longitude
alt = 0  # [m] Altitude
# Position (x, y, z) [m]
r_usr_ci = pnt.lat_lon_alt2cart(np.array([lat, lon, alt]), pnt.R_MOON)

print("User:", r_usr_ci)

<img src="https://raw.githubusercontent.com/Stanford-NavLab/LuPNT/development/examples/python/notebooks/ex_frozen_orbits/figures/coe.png" width="400"/> 

In [ ]:
# Time
t0 = pnt.gregorian2time(2024, 6, 5, 12, 30, 0)  # [s] Initial time
Dt = 10  # [s] Time step
tspan = np.arange(0, period, Dt)  # [s] Time span
tspan_h = tspan / pnt.SECS_HOUR  # [h] Time span
tfs = t0 + tspan  # [s] Time from start

In [ ]:
# Propagation
dyn = pnt.KeplerianDynamics(pnt.GM_MOON)
coe_ci = dyn.propagate(coe0_ci, t0, tfs)
rv_ci = pnt.classical_to_cart(coe_ci, pnt.GM_MOON)

# Frame plot
origin = np.zeros(3)
R_ci2mi = np.eye(3)

# Plot MI
fig = go.Figure()
pnt.plot.plot_body(fig, pnt.MOON)
pnt.plot.scatter(fig, r_usr_ci, marker_size=5, color="red")
pnt.plot.plot_frame(fig, origin, R_ci2mi, length=3 * pnt.R_MOON, width=5)
pnt.plot.plot_orbits(fig, rv_ci, t=0)
pnt.plot.plot_arrow3(fig, rv0_ci[:3], rv0_ci[3:], length=pnt.R_MOON, width=5)
pnt.plot.set_view(fig, azimuth=20, elevation=20, zoom=3)
fig.show()

In [ ]:
R_ci2enu = pnt.rot_x(pnt.PI_OVER_TWO - lat) @ pnt.rot_z(pnt.PI_OVER_TWO + lon)
enu = (rv_ci[:, :3] - r_usr_ci) @ R_ci2enu.T
rho = np.linalg.norm(enu, axis=1)
a = np.arctan2(enu[:, 1], enu[:, 0])
el = np.arcsin(enu[:, 2] / rho)

# az, el, rho = pnt.cart2az_el_range(rv_ci[:, :3], r_usr_ci).T

el_min = 15 * pnt.RAD  # [rad] Minimum elevation
idxs = el > el_min
print("Visible duration:", round(np.sum(idxs) * Dt / pnt.SECS_HOUR, 2), "hours")

plt.figure()
plt.plot(tspan_h, el / pnt.RAD)
plt.xlim(tspan_h[[0, -1]])
plt.xlabel("Time [h]")
plt.ylabel("Elevation [deg]")
plt.grid()
plt.show()